In [1]:
import numpy as np
import pandas as pd

rand = np.random.default_rng(42) # 랜덤 생성
n = 2000 # 2000 개의 제품

In [2]:
# 상류 온도 : 천천히 흔들리는 시계열
temp_1 = np.zeros(n) # 2000 개의 빈 배열 생성
temp_1[0] = 180 # 첫 온도를 180 으로 생성
for t in range(1, n):
    # 180 도를 기준으로 이전온도차 에 95% 를 유지하도록 한다.
    # 거기에 평균 0, 표준편차 1 의 난수를 생성하여 더한다.(대부분 -2~2 일 것임)
    temp_1[t] = 180 + 0.95 * (temp_1[t-1] - 180) + rand.normal(0, 1)

In [ ]:
# 위에서 만든 온도를 기준으로 여러 환경을 랜덤하게 추가
# 어떤 제품이 하나 만들어질때 임의의 센서값이 완성된다.
df = pd.DataFrame({
    'lot_no'   : np.arange(1, n + 1),       # 1 ~ 2000 까지 생성
    'temp_1'   : temp_1,                    # 1공정 온도(℃) 2000개 배열
    'press_1'  : rand.normal(5.0, 0.2, n),   # 1공정 압력(bar) n(2000)개 생성
    'time_1'   : rand.normal(60, 2, n),      # 1공정 유지시간(s) n(2000)개 생성
    'temp_2'   : rand.normal(120, 2, n),     # 2공정 온도(℃) n(2000)개 생성
    'amb_temp' : rand.normal(24, 2, n),      # 라인 온도(℃) n(2000)개 생성
    'amb_humid': rand.normal(50, 8, n),      # 라인 습도(%) n(2000)개 생성
})
df

,lot_no,temp_1,press_1,time_1,temp_2,amb_temp,amb_humid
0,1,180.000000,5.147758,60.152210,118.877018,22.438390,56.004269
1,2,180.304717,4.915500,59.779638,117.400366,21.977145,38.544877
2,3,179.249497,5.193910,59.583252,118.876093,23.905205,49.857152
3,4,180.037473,4.913470,59.317179,118.526775,21.307269,64.435130
4,5,180.976165,5.070994,59.760683,119.971689,22.569288,68.893044


In [7]:
# 치수 : 상류 온도는 3로트 뒤에 영향 (문제는 상류, 발견은 하류)
# 온도를 3칸 밀어준 값 생성, 그로 인해 빈 값은 180 으로 채워줌
temp_lag3 = df['temp_1'].shift(3).fillna(180)
"""[175][178][182][180][179] -> [NaN][NaN][NaN][175][178] -> [180][180][180][175][178]"""

'[175][178][182][180][179] -> [NaN][NaN][NaN][175][178] -> [180][180][180][175][178]'

In [8]:
# 최종 완성품의 치수 dim 에 영향을 주기 위해...
# 나머지는 임의의 값으로 채웠다.
df['dim'] = (10.0
             + 0.004  * (temp_lag3 - 180)
             - 0.03   * (df['press_1'] - 5.0)
             + 0.0008 * (df['time_1'] - 60) ** 2
             + 0.0005 * (df['amb_humid'] - 50)
             + 0.0002 * (temp_lag3 - 180) * (df['amb_humid'] - 50) / 8
             + rand.normal(0, 0.004, n))
 
# colab -> Colab CPU -> terminal
df.to_csv('/content/data/p03_process.csv', index=False)
df.head()

,lot_no,temp_1,press_1,time_1,temp_2,amb_temp,amb_humid,dim
0,1,180.000000,5.442831,57.493389,117.234477,24.496788,48.537201,9.985421
1,2,180.304717,4.909610,60.506409,120.450233,24.667205,51.410216,10.002383
2,3,179.249497,4.866824,61.790437,119.893314,26.472686,57.192988,10.010397
3,4,180.037473,5.086802,60.546643,121.728087,26.431659,38.221835,9.986783
4,5,180.976165,5.050371,64.477663,121.213466,23.098086,47.717615,10.009038
